# NB09e: Landuse Plausibility Filter (v8 — EXPERIMENTAL)## Pipeline positionNB05a/b → NB06–NB09a-d → **NB09e** → NB10 (future: apply filter on winning models)## PurposeEvaluate plausibility signals **directly against UNOSAT ground truth** (damage_binary),independent of any ML model. This answers: "do landuse transition, NDVI loss,COH drop, and building size independently predict damage?"Future NB10 series will apply this filter on top of the best models selected via NB13.## Plausibility signals1. Landuse transition (built_up → bare_soil = plausible damage)2. NDVI change (vegetation loss = destruction)3. Building size (< 50 m² unreliable at 10m Sentinel resolution)4. Coherence drop (pre-post COH decrease = structural change)5. Combined plausibility score (weighted sum of 1–4)## Data sources- Parquets: bda_product_prepost + bda_buildings (per-tier)- NB13 consolidated_experiments CSV (for cross-model context)- NB08a tier_sensor_ablation CSV (best ablation results)- NB08b comparison tables CSV (xBD diagnostic results)## Key design decisionPrevious versions (v1–v7) evaluated plausibility as a post-processing filter on NB09a RFpredictions. This was flawed: NB09a is one of the weakest models (sometimes below random),making the filter evaluation model-dependent and unreliable. v8 evaluates plausibilitysignals as standalone damage predictors against UNOSAT labels.---

> Copyright (C) 2024-2026 Marco Heinzen - SPDX-License-Identifier: AGPL-3.0-or-later
> Part of the Master Thesis "Building Damage Assessment with Multimodal Satellite Time Series and Machine Learning in the Russia-Ukraine War 2022-2026"
> Code hosted at https://github.com/marcoheinzen/bda
> Parts of this code were written or improved with the assistance of Claude (Anthropic); all other code, and the concept, research, architecture, design, execution, testing and validation throughout, are the author's work.


# CELL 3: NB09e CONFIG + GLOBAL SETUP

In [1]:
# @title CELL 3: NB09e CONFIG + GLOBAL SETUP
TIER_SELECTION = [0,1,2]
CITY_SELECTION = None
REQUIRE_UNOSAT = True

import platform, os
os.environ['PROJ_DATA'] = os.path.expanduser('~/miniconda3/envs/bda/share/proj')
os.environ['PROJ_LIB'] = os.environ['PROJ_DATA']
import pyproj
pyproj.datadir.set_data_dir(os.environ['PROJ_DATA'])

if platform.system() == 'Windows':
    _setup = r'F:\PROJECTS\masterthesis\gdrive\masterthesis\notebooks\global_setup.py'
elif os.path.exists('/content/drive_f'):
    _setup = '/content/drive_f/masterthesis/notebooks/global_setup.py'
else:
    _setup = '/mnt/f/PROJECTS/masterthesis/gdrive/masterthesis/notebooks/global_setup.py'
with open(_setup) as f:
    exec(f.read())


/home/alpineobotics/miniconda3/envs/bda/lib/python3.12/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


BDA GLOBAL SETUP
Started: 2026-04-12 21:56:49
Python: 3.12.12

[1/7] Directory Structure
----------------------------------------------------------------------
  GDrive (G:):       /content/drive_f/masterthesis OK
  GDrive (F:):       /content/drive_f/masterthesis OK
  Local data (G:):   /content/masterthesis_local/data OK
  Data stack (F:):   /mnt/f/PROJECTS/masterthesis/data_stack OK

  TIER_SELECTION: [0, 1, 2]
  CITY_SELECTION: None (tier filter)
  REQUIRE_UNOSAT: True
  CITIES_TO_PROCESS: 21 cities

[2/7] Credentials
----------------------------------------------------------------------
  Copernicus: inf***
  OpenTopography: OK
  Earthdata: marcoheinzen

[3/7] Python Packages
----------------------------------------------------------------------


<string>:536: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.



  Already installed: 23
  Newly installed:   0
  Failed:            0

[4/7] Global Imports & Configuration
----------------------------------------------------------------------
  All imports loaded

[5/7] Processing Config & SNAP
----------------------------------------------------------------------
  GPT: Usage:
  Temporal baseline: 10-24 days
  Wavelength: 0.0555

[6/7] GPU Status
----------------------------------------------------------------------
  CUDA available: NVIDIA GeForce RTX 2070 SUPER
    CUDA version: 12.8

[7/7] Disk Space
----------------------------------------------------------------------
  GDrive (G:)     913.0/7452.0 GB (6539.0 GB free)
  GDrive (F:)     1177.8/3726.0 GB (2548.2 GB free)
  Local data      11361.5/14901.9 GB (3540.3 GB free)
  Data stack      1177.8/3726.0 GB (2548.2 GB free)
  WSL ext4        103.0/1006.9 GB (852.6 GB free)

GLOBAL SETUP COMPLETE
  Torch device: cuda
  Cities: 21, CITY=Avdiivka
  Functions: load_aoi(), load_aoi_gdf(), load_aoi

# CELL S0: LOAD PARQUET + NB13/NB08 RESULTS

In [2]:
# @title CELL S0: LOAD PARQUET + NB13/NB08 RESULTS
import sys, importlib
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score

print("=" * 70)
print("CELL S0: LOAD PARQUET + NB13/NB08 RESULTS")
print("=" * 70)

CITIES_TO_PROCESS, _battle_dates = resolve_cities(
    tier_selection=TIER_SELECTION,
    city_selection=[CITY_SELECTION] if isinstance(CITY_SELECTION, str) else CITY_SELECTION,
    require_unosat=REQUIRE_UNOSAT,
)

_nb_dir = str(NOTEBOOKS_DIR)
if _nb_dir not in sys.path:
    sys.path.insert(0, _nb_dir)

import stack_loader
importlib.reload(stack_loader)
from stack_loader import get_feature_groups

# ---- load per-tier parquets (consistent with NB09a-c) ----
_tiers = TIER_SELECTION if isinstance(TIER_SELECTION, list) else [0, 1, 2]
df_pp = load_tier_parquets(PARQUET_PREPOST_TIER_FMT, _tiers)
df_bldg = load_tier_parquets(PARQUET_BUILDINGS_TIER_FMT, _tiers)

join_cols = ['building_id', 'city']
bldg_extra = [c for c in df_bldg.columns if c not in df_pp.columns]
df = df_pp.merge(df_bldg[join_cols + bldg_extra], on=join_cols, how='left')
del df_pp
df = df[df['city'].isin(CITIES_TO_PROCESS)].copy()
df = df[df['damage_binary'] >= 0].copy()
print(f"  Loaded: {len(df)} buildings, {len(df.columns)} columns, {df['city'].nunique()} cities")

feature_groups = get_feature_groups(df)
TARGET_COL = 'damage_binary'

# ---- load NB13 consolidated experiments CSV (latest) ----
nb13_dir = RESULTS_ROOT / 'nb13'
nb13_csv = sorted(nb13_dir.glob('consolidated_experiments_*.csv')) if nb13_dir.exists() else []
NB13_DF = None
if nb13_csv:
    NB13_DF = pd.read_csv(nb13_csv[-1])
    print(f"  NB13 consolidated: {nb13_csv[-1].name} ({len(NB13_DF)} experiments)")
    if 'auc' in NB13_DF.columns:
        best = NB13_DF.loc[NB13_DF['auc'].idxmax()]
        print(f"    Best overall: {best.get('experiment_name','?')} AUC={best['auc']:.4f} ({best.get('notebook','?')})")
else:
    print(f"  NB13 consolidated CSV not found at {nb13_dir}")
    print(f"  Run NB13 first. L3 context comparison will be skipped.")

# ---- load NB08a ablation results (latest) ----
nb08a_dir = RESULTS_ROOT / 'nb08a'
nb08a_csv = sorted((nb08a_dir / 'cell_ablation').glob('tier_sensor_ablation_*.csv')) if (nb08a_dir / 'cell_ablation').exists() else []
NB08A_ABLATION = None
if nb08a_csv:
    NB08A_ABLATION = pd.read_csv(nb08a_csv[-1])
    print(f"  NB08a ablation: {nb08a_csv[-1].name} ({len(NB08A_ABLATION)} rows)")
else:
    print(f"  NB08a ablation CSV not found")

# ---- load NB08b comparison tables (latest) ----
nb08b_dir = RESULTS_ROOT / 'nb08b'
nb08b_csv = sorted(nb08b_dir.glob('*.csv')) if nb08b_dir.exists() else []
NB08B_DF = None
if nb08b_csv:
    NB08B_DF = pd.read_csv(nb08b_csv[-1])
    print(f"  NB08b results: {nb08b_csv[-1].name} ({len(NB08B_DF)} rows)")
else:
    print(f"  NB08b results CSV not found")

# identify landuse columns
lu_cols = {}
for gname, cols in feature_groups.items():
    if gname.startswith('lu_') or gname == 'landuse':
        for c in cols:
            if c in df.columns:
                lu_cols.setdefault(gname, []).append(c)

# identify NDVI columns from composites
ndvi_cols = [c for c in df.columns if 'ndvi' in c.lower()]
print(f"  NDVI columns found: {ndvi_cols[:5]}")

# building attribute columns
bldg_cols = [c for c in feature_groups.get('building_attrs', []) if c in df.columns]

# landuse mode columns (s2__landuse__*_mode)
lu_mode_cols = [c for c in df.columns if 'landuse' in c and 'mode' in c]
print(f"\n  Landuse mode columns: {lu_mode_cols}")
print(f"  Landuse groups: {len(lu_cols)} ({sum(len(v) for v in lu_cols.values())} columns)")
print(f"  NDVI columns: {len(ndvi_cols)}")
print(f"  Building attrs: {bldg_cols}")
print(f"  Cities: {sorted(df['city'].unique())}")


# ---- RESULT REGISTRY (for NB13 consolidation) ----
from bda_results import ResultRegistry
registry = ResultRegistry(RESULTS_ROOT, notebook='NB09e')


# ---- SAVE HELPERS ----
import matplotlib.pyplot as plt
from datetime import datetime as _dt

OUT_DIR = RESULTS_ROOT / 'nb09e'
OUT_DIR.mkdir(parents=True, exist_ok=True)

def save_result(data, name, cell_id, fmt='csv'):
    cell_dir = OUT_DIR / cell_id
    cell_dir.mkdir(parents=True, exist_ok=True)
    ts = _dt.now().strftime('%Y%m%d_%H%M%S')
    if fmt == 'csv' and isinstance(data, pd.DataFrame):
        path = cell_dir / f"{name}_{ts}.csv"
        data.to_csv(path, index=False)
    elif fmt == 'json':
        path = cell_dir / f"{name}_{ts}.json"
        import json as _j
        with open(path, 'w') as fh:
            _j.dump(data, fh, indent=2, default=str)
    else:
        raise ValueError(f"Unknown fmt={fmt}")
    print(f"  Saved: {path.relative_to(OUT_DIR)} ({path.stat().st_size / 1024:.1f} KB)")
    return path

def save_fig(fig, name, cell_id, dpi=150):
    cell_dir = OUT_DIR / cell_id
    cell_dir.mkdir(parents=True, exist_ok=True)
    ts = _dt.now().strftime('%Y%m%d_%H%M%S')
    path = cell_dir / f"{name}_{ts}.png"
    fig.savefig(path, dpi=dpi, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f"  Plot: {path.relative_to(OUT_DIR)}")
    return path

# ---- DATASET PROFILE (for NB13) ----
META_COLS = ['building_id', 'city', 'damage_binary', 'geometry']
FEATURE_COLS = [c for c in df.columns if c not in META_COLS]
registry.log_dataset_profile(
    parquet_name='bda_product_prepost',
    parquet_fmt=str(PARQUET_PREPOST_TIER_FMT),
    tier_selection=TIER_SELECTION if isinstance(TIER_SELECTION, list) else [0, 1, 2],
    df=df,
    feature_cols=FEATURE_COLS,
    meta_cols=META_COLS,
)
print(f"  Dataset profile logged to registry")

print(f"  Output: {OUT_DIR}")
print(f"  Helpers: save_result(), save_fig()")


CELL S0: LOAD PARQUET + NB13/NB08 RESULTS
  load_tier_parquets: 3 tiers, 907371 rows
  load_tier_parquets: 3 tiers, 907371 rows
  Loaded: 598595 buildings, 372 columns, 21 cities
  NB13 consolidated: consolidated_experiments_20260412_185831.csv (84 experiments)
    Best overall: BDA__RGB+CARD__LogReg AUC=0.7690 (NB08b)
  NB08a ablation CSV not found
  NB08b results: full_comparison.csv (72 rows)
  NDVI columns found: ['s2__composite__ndvi__post_winter_baseline_mean', 's2__composite__ndvi__post_winter_baseline_std', 's2__composite__ndvi__post_winter_baseline_max', 's2__composite__ndvi__prebattle_baseline_mean', 's2__composite__ndvi__prebattle_baseline_std']

  Landuse mode columns: ['s2__composite__landuse__post_winter_baseline_mode', 's2__composite__landuse__prebattle_baseline_mode', 's2__composite__landuse__winter_baseline_mode']
  Landuse groups: 0 (0 columns)
  NDVI columns: 9
  Building attrs: ['roof_height', 'num_floors', 'height', 'area_m2', 'n_pixels']
  Cities: ['Avdiivka', 'Bo

TypeError: ufunc 'divide' not supported for the input types, and the inputs could not be safely coerced to any supported types according to the casting rule ''safe''

# CELL L1: LANDUSE TRANSITION MATRIX

In [ ]:
# @title CELL L1: LANDUSE TRANSITION MATRIX
# =============================================================================
# Cross-tabulate pre-battle vs post-battle landuse class per building.
# Landuse classes (from NB03d): water, vegetation, bare_soil, built_up, snow/ice
# Damage indicator: built_up -> bare_soil or built_up -> vegetation (rubble + regrowth)
# =============================================================================

print("=" * 70)
print("CELL L1: LANDUSE TRANSITION MATRIX")
print("=" * 70)

# find pre and post landuse mode columns
pre_lu_mode = [c for c in df.columns if ('prebattle' in c or 'winter_baseline' in c) and 'landuse' in c and 'mode' in c and 'post' not in c]
post_lu_mode = [c for c in df.columns if ('post_winter' in c or 'postbattle' in c or 'assessment' in c) and 'landuse' in c and 'mode' in c]

# also check for crossbattle
cross_lu_mode = [c for c in df.columns if 'crossbattle' in c and 'landuse' in c and 'mode' in c]

print(f"  Pre-battle LU columns:  {pre_lu_mode}")
print(f"  Post-battle LU columns: {post_lu_mode}")
print(f"  Cross-battle LU columns: {cross_lu_mode}")

# NB03d class mapping (Cell 15, L644-645):
# 0=NoData, 1=Snow, 2=Water, 3=Vegetation, 4=Sparse Veg, 5=Urban, 6=Bare Soil, 7=Shadow
LANDUSE_NAMES = {0: 'nodata', 1: 'snow', 2: 'water', 3: 'vegetation', 4: 'sparse_veg', 5: 'urban', 6: 'bare_soil', 7: 'shadow'}

if pre_lu_mode and post_lu_mode:
    pre_col = pre_lu_mode[0]
    post_col = post_lu_mode[0]
    
    df_lu = df[['building_id', pre_col, post_col, TARGET_COL, 'city']].dropna().copy()
    df_lu['pre_class'] = df_lu[pre_col].round().astype(int).map(LANDUSE_NAMES).fillna('unknown')
    df_lu['post_class'] = df_lu[post_col].round().astype(int).map(LANDUSE_NAMES).fillna('unknown')
    df_lu['transition'] = df_lu['pre_class'] + ' -> ' + df_lu['post_class']
    
    # transition matrix: counts
    trans_matrix = pd.crosstab(df_lu['pre_class'], df_lu['post_class'], margins=True)
    print(f"\n  Transition matrix (all buildings):")
    print(trans_matrix.to_string())
    
    # transition matrix: damage rate
    print(f"\n  Damage rate by transition:")
    print(f"  {'Transition':35s} {'n':>6s} {'dmg':>5s} {'rate':>7s}")
    print(f"  {'-'*35} {'-'*6} {'-'*5} {'-'*7}")
    
    trans_stats = df_lu.groupby('transition').agg(
        n=(TARGET_COL, 'count'),
        n_damaged=(TARGET_COL, 'sum')
    ).reset_index()
    trans_stats['rate'] = trans_stats['n_damaged'] / trans_stats['n']
    trans_stats = trans_stats.sort_values('rate', ascending=False)
    
    for _, row in trans_stats.iterrows():
        print(f"  {row['transition']:35s} {row['n']:6.0f} {row['n_damaged']:5.0f} {row['rate']:7.3f}")
    
    # flag plausible damage transitions
# NB03d P4 lines 24-30: damage transitions
    PLAUSIBLE_DAMAGE = {'urban -> bare_soil', 'urban -> vegetation', 'urban -> sparse_veg'}
    IMPLAUSIBLE_DAMAGE = {'vegetation -> vegetation', 'water -> water', 'bare_soil -> bare_soil',
                              'sparse_veg -> sparse_veg', 'snow -> snow'}
    
    lu_merge = df_lu[['building_id', 'city', 'transition']].rename(columns={'transition': 'lu_transition'})
    df = df.merge(lu_merge, on=['building_id', 'city'], how='left')
    df['lu_plausible_damage'] = df['lu_transition'].isin(PLAUSIBLE_DAMAGE).astype(int)
    df['lu_implausible_damage'] = df['lu_transition'].isin(IMPLAUSIBLE_DAMAGE).astype(int)
    
    n_plausible = df['lu_plausible_damage'].sum()
    n_implausible = df['lu_implausible_damage'].sum()
    print(f"\n  Plausible damage transitions: {n_plausible}")
    print(f"  Implausible damage transitions: {n_implausible}")

    save_result(trans_stats, 'landuse_transitions', 'cell_l1')

    # register landuse plausible_damage as standalone binary signal
    lu_mask = df['lu_plausible_damage'].notna() & (df[TARGET_COL] >= 0)
    if lu_mask.sum() > 50 and len(df.loc[lu_mask, TARGET_COL].unique()) == 2:
        registry.log_experiment(
            cell_id='cell_l1',
            experiment_name='L1_landuse_transition_binary',
            parquet_name='bda_product_prepost',
            tier_selection=TIER_SELECTION if isinstance(TIER_SELECTION, list) else [0, 1, 2],
            classifier_name='signal_lu_plausible_damage',
            feature_set_name='lu_plausible_damage',
            feature_cols=['lu_plausible_damage'],
            cv_method='none',
            n_folds=0,
            imputation='none',
            y_true=df.loc[lu_mask, TARGET_COL].values,
            y_proba=df.loc[lu_mask, 'lu_plausible_damage'].values.astype(float),
            groups=df.loc[lu_mask, 'city'].values,
            note='Landuse plausible-damage transition as binary damage predictor',
            tags=['plausibility', 'landuse', 'standalone'],
        )
        print(f"  Registered lu_plausible_damage to registry")
else:
    print(f"  Cannot build transition matrix: need pre + post landuse mode columns")
    df['lu_plausible_damage'] = np.nan
    df['lu_implausible_damage'] = np.nan


# CELL L2: PLAUSIBILITY SCORING

In [ ]:
# @title CELL L2: PLAUSIBILITY SCORING
# =============================================================================
# Combine multiple plausibility signals into a single score per building:
#   1. Landuse transition (L1)
#   2. NDVI change (pre-post decrease = damage, increase = regrowth)
#   3. Building size (< 50 m2 unreliable at 10m)
#   4. Coherence drop (pre-post COH decrease = structural change)
# =============================================================================

print("=" * 70)
print("CELL L2: PLAUSIBILITY SCORING")
print("=" * 70)

# --- NDVI change ---
pre_ndvi = [c for c in ndvi_cols if ('prebattle' in c or 'winter_baseline' in c) and 'mean' in c and 'post' not in c]
post_ndvi = [c for c in ndvi_cols if ('post_winter' in c or 'postbattle' in c) and 'mean' in c]

if pre_ndvi and post_ndvi:
    df['ndvi_delta'] = df[post_ndvi[0]] - df[pre_ndvi[0]]
    print(f"  NDVI delta: {pre_ndvi[0]} -> {post_ndvi[0]}")
    print(f"  Range: [{df['ndvi_delta'].min():.3f}, {df['ndvi_delta'].max():.3f}]")
    
    # negative NDVI delta on built-up land = damage signal
    # positive NDVI delta = vegetation regrowth over rubble (also damage signal!)
    # zero = stable
else:
    df['ndvi_delta'] = np.nan
    print(f"  No pre/post NDVI columns found")

# --- Building size filter ---
if 'area_m2' in df.columns:
    df['bldg_reliable'] = (df['area_m2'] >= 50).astype(int)
    n_small = (df['area_m2'] < 50).sum()
    n_total = len(df)
    print(f"  Buildings < 50 m2: {n_small} ({100*n_small/n_total:.1f}%) -> flagged unreliable")
else:
    df['bldg_reliable'] = 1
    print(f"  No area_m2 column")

# --- Coherence drop ---
coh_bl_mean = [c for c in df.columns if 'coh' in c.lower() and ('baseline' in c.lower() or 'prebattle' in c.lower()) and 'mean' in c.lower()]
coh_as_mean = [c for c in df.columns if 'coh' in c.lower() and ('assessment' in c.lower() or 'post_baseline' in c.lower() or 'postbattle' in c.lower()) and 'mean' in c.lower()]

if coh_bl_mean and coh_as_mean:
    df['coh_drop'] = df[coh_bl_mean[0]] - df[coh_as_mean[0]]
    print(f"  COH drop: {coh_bl_mean[0]} - {coh_as_mean[0]}")
    print(f"  Range: [{df['coh_drop'].min():.3f}, {df['coh_drop'].max():.3f}]")
    print(f"  Positive = coherence decreased = structural change = plausible damage")

    # separability check
    dmg_coh = df.loc[df[TARGET_COL] == 1, 'coh_drop'].dropna()
    und_coh = df.loc[df[TARGET_COL] == 0, 'coh_drop'].dropna()
    if len(dmg_coh) > 10 and len(und_coh) > 10:
        from sklearn.metrics import roc_auc_score
        auc_coh = roc_auc_score(
            df.loc[df['coh_drop'].notna(), TARGET_COL],
            df.loc[df['coh_drop'].notna(), 'coh_drop']
        )
        print(f"  COH drop AUC (standalone): {auc_coh:.3f}")
        print(f"  Damaged median drop: {dmg_coh.median():.4f}")
        print(f"  Undamaged median drop: {und_coh.median():.4f}")
else:
    df['coh_drop'] = np.nan
    print(f"  No COH baseline/assessment mean columns found")

# --- Composite plausibility score ---
# Simple weighted sum (interpretable for thesis, not learned)
df['plausibility_score'] = 0.0

# landuse transition: +1 if plausible, -1 if implausible
if 'lu_plausible_damage' in df.columns:
    df['plausibility_score'] += df['lu_plausible_damage'].fillna(0)
    df['plausibility_score'] -= df['lu_implausible_damage'].fillna(0)

# NDVI decrease on built-up: +0.5 for damage-consistent change
if 'ndvi_delta' in df.columns:
    df['plausibility_score'] += np.where(df['ndvi_delta'] < -0.1, 0.5, 0)

# coherence drop: +0.5 for significant COH decrease (structural change)
if 'coh_drop' in df.columns and df['coh_drop'].notna().sum() > 0:
    coh_threshold = df['coh_drop'].quantile(0.75)  # top 25% drops
    df['plausibility_score'] += np.where(df['coh_drop'] > coh_threshold, 0.5, 0)

# building size: -0.5 for unreliable small buildings
df['plausibility_score'] -= 0.5 * (1 - df['bldg_reliable'])

print(f"\n  Plausibility score range: [{df['plausibility_score'].min():.1f}, {df['plausibility_score'].max():.1f}]")
print(f"  Score distribution:")
for val in sorted(df['plausibility_score'].unique()):
    n = (df['plausibility_score'] == val).sum()
    dmg_rate = df.loc[df['plausibility_score'] == val, TARGET_COL].mean()
    print(f"    score={val:+.1f}: n={n:6d}, damage_rate={dmg_rate:.3f}")


# CELL L2b: STANDALONE PLAUSIBILITY vs UNOSAT

In [ ]:
# @title CELL L2b: STANDALONE PLAUSIBILITY vs UNOSAT
# =============================================================================
# Evaluate each plausibility signal as a standalone damage predictor against
# UNOSAT damage_binary. No ML model involved. This is the scientifically
# meaningful result: "do these signals independently predict damage?"
# =============================================================================

from sklearn.metrics import (roc_auc_score, f1_score, precision_score,
                             recall_score, average_precision_score,
                             matthews_corrcoef, balanced_accuracy_score)
from scipy.stats import mannwhitneyu
import matplotlib.pyplot as plt

print("=" * 70)
print("CELL L2b: STANDALONE PLAUSIBILITY SIGNALS vs UNOSAT")
print("=" * 70)

df_eval = df[df[TARGET_COL] >= 0].copy()
y_true = df_eval[TARGET_COL].values

# define signals: (column, higher_means_damage, description)
SIGNALS = []
if 'plausibility_score' in df_eval.columns:
    SIGNALS.append(('plausibility_score', True, 'Combined plausibility (L2)'))
if 'lu_plausible_damage' in df_eval.columns:
    SIGNALS.append(('lu_plausible_damage', True, 'Landuse: plausible damage transition'))
if 'lu_implausible_damage' in df_eval.columns:
    SIGNALS.append(('lu_implausible_damage', False, 'Landuse: implausible damage transition'))
if 'ndvi_delta' in df_eval.columns and df_eval['ndvi_delta'].notna().sum() > 100:
    SIGNALS.append(('ndvi_delta', False, 'NDVI delta (negative = damage)'))
if 'coh_drop' in df_eval.columns and df_eval['coh_drop'].notna().sum() > 100:
    SIGNALS.append(('coh_drop', True, 'COH drop (positive = structural change)'))
if 'bldg_reliable' in df_eval.columns:
    SIGNALS.append(('bldg_reliable', True, 'Building reliable (area >= 50 m2)'))

if not SIGNALS:
    print("  No plausibility signals available. Run L1 + L2 first.")
else:
    print(f"\n  Evaluating {len(SIGNALS)} signals against UNOSAT damage_binary:")
    print(f"  n_buildings={len(df_eval)}, damage_rate={y_true.mean():.3f}\n")

    rows = []
    print(f"  {'Signal':40s} {'AUC':>7s} {'AP':>7s} {'MWU-p':>10s} {'dmg_med':>9s} {'und_med':>9s} {'d_med':>8s}")
    print(f"  {'-'*40} {'-'*7} {'-'*7} {'-'*10} {'-'*9} {'-'*9} {'-'*8}")

    for col, higher_is_damage, desc in SIGNALS:
        mask = df_eval[col].notna()
        if mask.sum() < 50:
            print(f"  {desc:40s} (skipped, n={mask.sum()})")
            continue

        vals = df_eval.loc[mask, col].values
        yt = df_eval.loc[mask, TARGET_COL].values

        if len(np.unique(yt)) < 2:
            continue

        # if lower values mean damage, flip for AUC
        score_for_auc = vals if higher_is_damage else -vals

        auc = roc_auc_score(yt, score_for_auc)
        ap = average_precision_score(yt, score_for_auc)

        dmg_vals = vals[yt == 1]
        und_vals = vals[yt == 0]
        stat, pval = mannwhitneyu(dmg_vals, und_vals, alternative='two-sided')
        dmg_med = np.median(dmg_vals)
        und_med = np.median(und_vals)
        delta_med = dmg_med - und_med

        print(f"  {desc:40s} {auc:7.3f} {ap:7.3f} {pval:10.2e} {dmg_med:9.4f} {und_med:9.4f} {delta_med:+8.4f}")

        rows.append({
            'signal': col, 'description': desc,
            'higher_is_damage': higher_is_damage,
            'auc': round(auc, 4), 'avg_precision': round(ap, 4),
            'mwu_pvalue': pval,
            'damaged_median': round(dmg_med, 4),
            'undamaged_median': round(und_med, 4),
            'delta_median': round(delta_med, 4),
            'n_valid': int(mask.sum()),
        })

        registry.log_experiment(
            cell_id='cell_l2b',
            experiment_name=f'plausibility_standalone_{col}',
            parquet_name='bda_product_prepost',
            tier_selection=TIER_SELECTION if isinstance(TIER_SELECTION, list) else [0, 1, 2],
            classifier_name=f'signal_{col}',
            feature_set_name=col,
            feature_cols=[col],
            cv_method='none',
            n_folds=0,
            imputation='none',
            y_true=yt,
            y_proba=score_for_auc,
            groups=df_eval.loc[mask, 'city'].values,
            note=f'Standalone plausibility signal vs UNOSAT: {desc}',
            tags=['plausibility', 'standalone', 'experimental'],
        )

    if rows:
        signal_df = pd.DataFrame(rows)
        save_result(signal_df, 'plausibility_standalone_vs_unosat', 'cell_l2b')

        # ---- per-city breakdown for combined score ----
        if 'plausibility_score' in df_eval.columns:
            print(f"\n  Per-city AUC for combined plausibility_score:")
            print(f"  {'City':25s} {'AUC':>7s} {'n':>7s} {'dmg%':>7s}")
            print(f"  {'-'*25} {'-'*7} {'-'*7} {'-'*7}")
            city_rows = []
            for city in sorted(df_eval['city'].unique()):
                cdf = df_eval[df_eval['city'] == city]
                cm = cdf['plausibility_score'].notna()
                if cm.sum() < 20 or len(cdf.loc[cm, TARGET_COL].unique()) < 2:
                    continue
                c_auc = roc_auc_score(cdf.loc[cm, TARGET_COL], cdf.loc[cm, 'plausibility_score'])
                c_n = cm.sum()
                c_rate = cdf.loc[cm, TARGET_COL].mean()
                print(f"  {city:25s} {c_auc:7.3f} {c_n:7d} {100*c_rate:6.1f}%")
                city_rows.append({'city': city, 'auc': round(c_auc, 4), 'n': int(c_n), 'damage_rate': round(c_rate, 4)})
            if city_rows:
                save_result(pd.DataFrame(city_rows), 'plausibility_per_city_auc', 'cell_l2b')

    # ---- distribution + ROC plot ----
    if 'plausibility_score' in df_eval.columns:
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))

        ax = axes[0]
        dmg_s = df_eval.loc[df_eval[TARGET_COL] == 1, 'plausibility_score'].dropna()
        und_s = df_eval.loc[df_eval[TARGET_COL] == 0, 'plausibility_score'].dropna()
        ax.hist(und_s, bins=30, alpha=0.6, label=f'Undamaged (n={len(und_s)})', density=True)
        ax.hist(dmg_s, bins=30, alpha=0.6, label=f'Damaged (n={len(dmg_s)})', density=True)
        ax.set_xlabel('Plausibility Score')
        ax.set_ylabel('Density')
        ax.set_title('Plausibility Score vs UNOSAT Label')
        ax.legend()

        # ROC curve for combined score
        from sklearn.metrics import roc_curve
        ax2 = axes[1]
        mask_ps = df_eval['plausibility_score'].notna()
        if mask_ps.sum() > 50:
            fpr, tpr, _ = roc_curve(df_eval.loc[mask_ps, TARGET_COL], df_eval.loc[mask_ps, 'plausibility_score'])
            auc_ps = roc_auc_score(df_eval.loc[mask_ps, TARGET_COL], df_eval.loc[mask_ps, 'plausibility_score'])
            ax2.plot(fpr, tpr, label=f'Plausibility (AUC={auc_ps:.3f})')
            ax2.plot([0, 1], [0, 1], 'k--', alpha=0.3)
            ax2.set_xlabel('FPR')
            ax2.set_ylabel('TPR')
            ax2.set_title('ROC: Plausibility Score as Damage Predictor')
            ax2.legend()

        plt.tight_layout()
        save_fig(fig, 'plausibility_standalone_roc', 'cell_l2b')


# CELL L3: PLAUSIBILITY vs ALL MODELS (NB13 CONTEXT)

In [ ]:
# @title CELL L3: PLAUSIBILITY vs ALL MODELS (NB13 CONTEXT)
# =============================================================================
# Compare standalone plausibility AUC to ALL experiments from NB13 registry.
# Shows where plausibility sits relative to ML models, without depending on
# any single model. This is context, not a filter evaluation.
# =============================================================================

print("=" * 70)
print("CELL L3: PLAUSIBILITY vs ALL MODELS (NB13 CONTEXT)")
print("=" * 70)

# plausibility AUC from L2b
plaus_auc = None
if 'plausibility_score' in df_eval.columns:
    mask_ps = df_eval['plausibility_score'].notna() & (df_eval[TARGET_COL] >= 0)
    if mask_ps.sum() > 50 and len(df_eval.loc[mask_ps, TARGET_COL].unique()) == 2:
        plaus_auc = roc_auc_score(df_eval.loc[mask_ps, TARGET_COL], df_eval.loc[mask_ps, 'plausibility_score'])
        print(f"  Plausibility score AUC (standalone vs UNOSAT): {plaus_auc:.4f}")

if NB13_DF is not None and 'auc' in NB13_DF.columns:
    df_models = NB13_DF[NB13_DF['auc'].notna()].copy()
    # exclude reference rows
    if 'is_reference' in df_models.columns:
        df_models = df_models[~df_models['is_reference'].fillna(False).astype(bool)]

    print(f"\n  NB13 model landscape ({len(df_models)} experiments with AUC):")
    print(f"  {'Notebook':10s} {'Experiment':45s} {'AUC':>7s} {'Classifier':>15s}")
    print(f"  {'-'*10} {'-'*45} {'-'*7} {'-'*15}")

    for _, row in df_models.sort_values('auc', ascending=False).head(20).iterrows():
        marker = ""
        if plaus_auc is not None and row['auc'] < plaus_auc:
            marker = " << BELOW plausibility"
        print(f"  {str(row.get('notebook','')):10s} {str(row.get('experiment_name','')):45s} "
              f"{row['auc']:7.4f} {str(row.get('classifier_name','')):>15s}{marker}")

    if plaus_auc is not None:
        n_below = (df_models['auc'] < plaus_auc).sum()
        n_above = (df_models['auc'] > plaus_auc).sum()
        pct_below = 100 * n_below / len(df_models) if len(df_models) > 0 else 0
        print(f"\n  Plausibility score (AUC={plaus_auc:.4f}) beats {n_below}/{len(df_models)} "
              f"({pct_below:.0f}%) of registered ML experiments")
        print(f"  Models above plausibility: {n_above}")

    # per-notebook best
    print(f"\n  Best per notebook vs plausibility:")
    for nb_name in sorted(df_models['notebook'].dropna().unique()):
        nb_sub = df_models[df_models['notebook'] == nb_name]
        if nb_sub['auc'].notna().any():
            best_row = nb_sub.loc[nb_sub['auc'].idxmax()]
            delta = ""
            if plaus_auc is not None:
                d = best_row['auc'] - plaus_auc
                delta = f"  (vs plausibility: {d:+.4f})"
            print(f"    {nb_name:10s}: AUC={best_row['auc']:.4f} ({best_row.get('experiment_name','?')}){delta}")

    save_result(df_models[['notebook', 'experiment_name', 'auc', 'f1', 'classifier_name',
                           'feature_set_name', 'cv_method']].head(30),
                'model_landscape_top30', 'cell_l3')
else:
    print("  NB13 consolidated CSV not available. Run NB13 first.")
    print("  Skipping model landscape comparison.")

# ---- NB08a ablation context ----
if NB08A_ABLATION is not None and 'auc' in NB08A_ABLATION.columns:
    print(f"\n  NB08a ablation context:")
    print(f"  {'Tier':15s} {'Sensor':10s} {'AUC':>7s}")
    print(f"  {'-'*15} {'-'*10} {'-'*7}")
    for _, row in NB08A_ABLATION.sort_values('auc', ascending=False).head(10).iterrows():
        print(f"  {str(row.get('tier','')):15s} {str(row.get('sensor','')):10s} {row['auc']:7.3f}")
    best_abl = NB08A_ABLATION['auc'].max()
    if plaus_auc is not None:
        print(f"\n  Best ablation AUC={best_abl:.4f} vs plausibility AUC={plaus_auc:.4f} "
              f"(delta={best_abl - plaus_auc:+.4f})")

print(f"\n  NOTE: This cell provides CONTEXT only. NB10 will apply plausibility")
print(f"  as post-processing on the winning models selected via NB13.")


# CELL L4: NDVI CHANGE ANALYSIS

In [ ]:
# @title CELL L4: NDVI CHANGE ANALYSIS
# =============================================================================
# Vegetation change as independent damage signal:
# - NDVI decrease = destruction (buildings removed, bare soil exposed)
# - NDVI increase post-battle = vegetation reclaiming rubble (delayed signal)
# =============================================================================

import matplotlib.pyplot as plt

print("=" * 70)
print("CELL L4: NDVI CHANGE ANALYSIS")
print("=" * 70)

if 'ndvi_delta' in df.columns and df['ndvi_delta'].notna().sum() > 100:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    dmg = df[df[TARGET_COL] == 1]['ndvi_delta'].dropna()
    und = df[df[TARGET_COL] == 0]['ndvi_delta'].dropna()
    
    ax = axes[0]
    ax.hist(und, bins=50, alpha=0.6, label=f'Undamaged (n={len(und)})', density=True)
    ax.hist(dmg, bins=50, alpha=0.6, label=f'Damaged (n={len(dmg)})', density=True)
    ax.axvline(0, color='k', linestyle=':', alpha=0.5)
    ax.set_xlabel('NDVI delta (post - pre)')
    ax.set_ylabel('Density')
    ax.set_title('NDVI Change by Damage Class')
    ax.legend()
    
    # per-city NDVI delta
    ax2 = axes[1]
    cities = sorted(df['city'].unique())
    city_dmg_ndvi = []
    city_und_ndvi = []
    for city in cities:
        cdf = df[df['city'] == city]
        d = cdf[cdf[TARGET_COL] == 1]['ndvi_delta'].median()
        u = cdf[cdf[TARGET_COL] == 0]['ndvi_delta'].median()
        city_dmg_ndvi.append(d)
        city_und_ndvi.append(u)
    
    x = range(len(cities))
    ax2.barh([f"{c} (dmg)" for c in cities], city_dmg_ndvi, color='red', alpha=0.6, label='Damaged')
    ax2.barh([f"{c} (und)" for c in cities], city_und_ndvi, color='blue', alpha=0.6, label='Undamaged')
    ax2.axvline(0, color='k', linestyle=':')
    ax2.set_xlabel('Median NDVI delta')
    ax2.set_title('Per-City NDVI Change')
    ax2.legend()
    
    plt.tight_layout()
    save_fig(fig, 'ndvi_change_analysis', 'cell_l4')
    
    # separability
    from scipy.stats import mannwhitneyu
    if len(dmg) > 10 and len(und) > 10:
        stat, pval = mannwhitneyu(dmg, und, alternative='two-sided')
        print(f"  Mann-Whitney U: stat={stat:.0f}, p={pval:.2e}")
        print(f"  Damaged median NDVI delta: {dmg.median():.4f}")
        print(f"  Undamaged median NDVI delta: {und.median():.4f}")

        # register NDVI delta as standalone signal (negative = damage -> flip sign)
        ndvi_mask = df['ndvi_delta'].notna() & (df[TARGET_COL] >= 0)
        if ndvi_mask.sum() > 50 and len(df.loc[ndvi_mask, TARGET_COL].unique()) == 2:
            registry.log_experiment(
                cell_id='cell_l4',
                experiment_name='L4_ndvi_delta_standalone',
                parquet_name='bda_product_prepost',
                tier_selection=TIER_SELECTION if isinstance(TIER_SELECTION, list) else [0, 1, 2],
                classifier_name='signal_ndvi_delta',
                feature_set_name='ndvi_delta',
                feature_cols=['ndvi_delta'],
                cv_method='none',
                n_folds=0,
                imputation='none',
                y_true=df.loc[ndvi_mask, TARGET_COL].values,
                y_proba=-df.loc[ndvi_mask, 'ndvi_delta'].values,
                groups=df.loc[ndvi_mask, 'city'].values,
                note='NDVI delta as standalone damage predictor (sign-flipped: negative delta = damage)',
                tags=['plausibility', 'ndvi', 'standalone'],
            )
            print(f"  Registered ndvi_delta to registry")
else:
    print(f"  No NDVI delta data. Skipping.")


# CELL L5: GROUND TRUTH PLAUSIBILITY ANALYSIS

In [ ]:
# @title CELL L5: GROUND TRUTH PLAUSIBILITY ANALYSIS
# =============================================================================
# Analyze plausibility characteristics of UNOSAT ground truth labels directly.
# No ML model involved. Questions answered:
#   - What fraction of UNOSAT-damaged buildings have plausible characteristics?
#   - What fraction of UNOSAT-undamaged buildings look implausible if flagged?
#   - Which cities have the most plausibility-inconsistent labels?
# This informs the UNOSAT limitation discussion in the thesis.
# =============================================================================

print("=" * 70)
print("CELL L5: GROUND TRUTH PLAUSIBILITY ANALYSIS")
print("=" * 70)

df_gt = df[df[TARGET_COL] >= 0].copy()
damaged = df_gt[df_gt[TARGET_COL] == 1]
undamaged = df_gt[df_gt[TARGET_COL] == 0]

print(f"  Total: {len(df_gt)} buildings ({len(damaged)} damaged, {len(undamaged)} undamaged)")

# ---- UNOSAT-damaged buildings with implausible characteristics ----
print(f"\n  DAMAGED buildings (UNOSAT=1) with IMPLAUSIBLE characteristics:")
n_checks = 0

if 'lu_transition' in damaged.columns:
    dmg_implausible = damaged['lu_implausible_damage'].sum() if 'lu_implausible_damage' in damaged.columns else 0
    dmg_plausible = damaged['lu_plausible_damage'].sum() if 'lu_plausible_damage' in damaged.columns else 0
    print(f"    Landuse: {int(dmg_plausible)} plausible transitions, {int(dmg_implausible)} implausible")
    print(f"    -> {100*dmg_implausible/len(damaged):.1f}% of damaged buildings have implausible landuse transition")

    print(f"\n    Top landuse transitions for DAMAGED buildings:")
    trans_dmg = damaged['lu_transition'].value_counts().head(10)
    for trans, count in trans_dmg.items():
        pct = 100 * count / len(damaged)
        print(f"      {trans:35s}: {count:5d} ({pct:.1f}%)")
    n_checks += 1

if 'ndvi_delta' in damaged.columns and damaged['ndvi_delta'].notna().sum() > 10:
    dmg_ndvi_pos = (damaged['ndvi_delta'] > 0.05).sum()
    print(f"\n    NDVI: {dmg_ndvi_pos} damaged buildings have NDVI INCREASE (>{0.05})")
    print(f"    -> {100*dmg_ndvi_pos/len(damaged):.1f}% unexpected (vegetation grew on damaged site?)")
    n_checks += 1

if 'coh_drop' in damaged.columns and damaged['coh_drop'].notna().sum() > 10:
    dmg_coh_neg = (damaged['coh_drop'] < 0).sum()
    coh_valid = damaged['coh_drop'].notna().sum()
    print(f"\n    COH: {dmg_coh_neg}/{coh_valid} damaged buildings have COH INCREASE (no structural change)")
    print(f"    -> {100*dmg_coh_neg/coh_valid:.1f}% inconsistent with structural damage")
    n_checks += 1

if 'area_m2' in damaged.columns:
    dmg_small = (damaged['area_m2'] < 50).sum()
    print(f"\n    Size: {dmg_small} damaged buildings < 50 m2 ({100*dmg_small/len(damaged):.1f}%)")
    print(f"    -> unreliable at 10m resolution, damage assessment questionable")
    n_checks += 1

# ---- UNOSAT-undamaged buildings with damage-like characteristics ----
print(f"\n  UNDAMAGED buildings (UNOSAT=0) with DAMAGE-LIKE characteristics:")

if 'lu_plausible_damage' in undamaged.columns:
    und_plausible = undamaged['lu_plausible_damage'].sum()
    print(f"    Landuse: {int(und_plausible)} undamaged buildings have plausible-damage transitions")
    print(f"    -> {100*und_plausible/len(undamaged):.1f}% potential UNOSAT false negatives (Aimaiti 2022)")

if 'ndvi_delta' in undamaged.columns and undamaged['ndvi_delta'].notna().sum() > 10:
    und_ndvi_drop = (undamaged['ndvi_delta'] < -0.1).sum()
    print(f"    NDVI: {und_ndvi_drop} undamaged buildings have significant NDVI drop")
    print(f"    -> {100*und_ndvi_drop/len(undamaged):.1f}% potential missed damage")

if 'coh_drop' in undamaged.columns and undamaged['coh_drop'].notna().sum() > 10:
    coh_thresh = df_gt['coh_drop'].quantile(0.75)
    und_coh_high = (undamaged['coh_drop'] > coh_thresh).sum()
    coh_valid = undamaged['coh_drop'].notna().sum()
    print(f"    COH: {und_coh_high}/{coh_valid} undamaged buildings have high COH drop (>p75={coh_thresh:.3f})")
    print(f"    -> {100*und_coh_high/coh_valid:.1f}% potential missed damage")

# ---- per-city plausibility consistency ----
if 'plausibility_score' in df_gt.columns:
    print(f"\n  Per-city plausibility consistency:")
    print(f"  {'City':25s} {'n_dmg':>7s} {'n_und':>7s} {'dmg_plaus':>10s} {'und_plaus':>10s} {'consist%':>9s}")
    print(f"  {'-'*25} {'-'*7} {'-'*7} {'-'*10} {'-'*10} {'-'*9}")
    city_consist = []
    for city in sorted(df_gt['city'].unique()):
        cdf = df_gt[df_gt['city'] == city]
        c_dmg = cdf[cdf[TARGET_COL] == 1]
        c_und = cdf[cdf[TARGET_COL] == 0]
        # damaged with positive plausibility = consistent
        dmg_consistent = (c_dmg['plausibility_score'] > 0).sum() if len(c_dmg) > 0 else 0
        # undamaged with non-positive plausibility = consistent
        und_consistent = (c_und['plausibility_score'] <= 0).sum() if len(c_und) > 0 else 0
        total = len(c_dmg) + len(c_und)
        consist_pct = 100 * (dmg_consistent + und_consistent) / total if total > 0 else 0
        print(f"  {city:25s} {len(c_dmg):7d} {len(c_und):7d} "
              f"{dmg_consistent:10d} {und_consistent:10d} {consist_pct:8.1f}%")
        city_consist.append({
            'city': city, 'n_damaged': len(c_dmg), 'n_undamaged': len(c_und),
            'dmg_consistent': dmg_consistent, 'und_consistent': und_consistent,
            'consistency_pct': round(consist_pct, 1),
        })
    if city_consist:
        save_result(pd.DataFrame(city_consist), 'plausibility_consistency_per_city', 'cell_l5')

if n_checks > 0:
    print(f"\n  THESIS IMPLICATION: UNOSAT labels have measurable inconsistencies with")
    print(f"  physical plausibility signals. This supports the Aimaiti 2022 finding that")
    print(f"  UNOSAT underreports damage. The plausibility score can flag potential")
    print(f"  false negatives for manual review in NB10 operational pipeline.")


# CELL L6: PLAUSIBILITY FILTER SUMMARY

In [ ]:
# @title CELL L6: PLAUSIBILITY FILTER SUMMARY
print("=" * 70)
print("NB09e PLAUSIBILITY FILTER SUMMARY (EXPERIMENTAL v8)")
print("=" * 70)

print(f"\n  Plausibility signals evaluated against UNOSAT ground truth:")
print(f"    1. Landuse transition (pre->post class change)")
print(f"    2. NDVI change (vegetation proxy for destruction)")
print(f"    3. Building size (< 50 m2 unreliable at 10m)")
print(f"    4. Coherence drop (pre-post COH decrease)")
print(f"    5. Combined plausibility score (weighted sum of 1-4)")
print(f"\n  Key results:")
print(f"    - Standalone AUC vs UNOSAT: see L2b table")
print(f"    - Per-city consistency: see L5 table")
print(f"    - Context vs ML models: see L3 table")

print(f"\n  DESIGN DECISION (v8 vs v1-v7):")
print(f"    Previous versions evaluated plausibility as post-processing on NB09a RF predictions.")
print(f"    This was flawed: NB09a is one of the weakest models, making the evaluation")
print(f"    model-dependent. v8 evaluates against UNOSAT directly.")
print(f"\n  FUTURE (NB10 series):")
print(f"    NB10 will take the BEST models from NB13 + Optuna tuning,")
print(f"    then apply plausibility as post-processing filter on those winning models.")
print(f"    That evaluation belongs in NB10, not NB09e.")

# ---- SAVE REGISTRY ----
registry.save()
print(f"\n  Registry saved to {RESULTS_ROOT / 'registry'}")
